# Exploring Confluent Flink SQL with Jupyter

This notebook demonstrates how to use `confluent-sql` to interactively explore
Flink tables (Kafka topics) from a Jupyter notebook.

## Setup

```bash
uv add jupyterlab pandas confluent-sql
```

Set your environment variables before starting Jupyter:
- `CONFLUENT_FLINK_API_KEY`
- `CONFLUENT_FLINK_API_SECRET`
- `CONFLUENT_ENV_ID`
- `CONFLUENT_ORG_ID`
- `CONFLUENT_COMPUTE_POOL_ID`
- `CONFLUENT_CLOUD_PROVIDER`
- `CONFLUENT_CLOUD_REGION`
- `CONFLUENT_TEST_DBNAME` (optional)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from _connection import get_connection

conn = get_connection()
print("Connected to Confluent Flink SQL")

## List available tables

Tables in Flink SQL correspond to Kafka topics in your cluster.

In [ ]:
tables = pd.read_sql("SHOW TABLES", conn)
tables

## Describe a table schema

In [ ]:
# Replace 'orders' with a table name from the list above
schema = pd.read_sql("DESCRIBE `orders`", conn)
schema

## Query data

Snapshot queries return point-in-time results, just like a traditional database.

In [ ]:
df = pd.read_sql("SELECT * FROM orders LIMIT 100", conn)
df.head(10)

In [ ]:
df.describe()

## Visualize

In [ ]:
# Example: plot a histogram if there's a numeric column
numeric_cols = df.select_dtypes(include="number").columns.tolist()
if numeric_cols:
    df[numeric_cols[0]].hist(bins=20)
else:
    print("No numeric columns to plot")

## Aggregation queries

Flink SQL supports standard SQL aggregations over your Kafka topics.

In [ ]:
agg = pd.read_sql(
    """
    SELECT
        customer_id,
        COUNT(*) AS order_count,
        SUM(amount) AS total_amount
    FROM orders
    GROUP BY customer_id
    """,
    conn,
)
agg.sort_values("total_amount", ascending=False).head(10)

## Cleanup

In [ ]:
conn.close()
print("Connection closed.")